# 01 — Data Exploration
**Wildfire Risk Modeling Exercise | Cell2Fire W — Scott & Burgan**

Inspect all raw inputs before any transformation. No data is modified here.

| Step | What it does | Report section |
|---|---|---|
| Sanity check | Verify all raw files exist | — |
| Raster metadata | CRS, shape, resolution, value range | Inputs Used |
| Weather & ignition | Station summary, wind/RH/temp, ignition coords | Inputs Used |
| Buildings | Feature count, CRS, attribute columns | Inputs Used |
| Visualisation | 12-panel figure of all raw layers | Inputs Used |

In [ ]:
# ── Town selection — change this to run Prairie
TOWN = "forest"   # "forest" or "prairie"

In [ ]:
# ============================================================
# CELL 1 — Imports, config, paths, sanity check
# ============================================================
import sys, pathlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import yaml
from scipy.ndimage import sobel

REPO_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils import load_config

cfg   = load_config(TOWN, REPO_ROOT)
RAW   = REPO_ROOT / cfg["data_raw"]
PLOTS = REPO_ROOT / cfg["plots_dir"]
PLOTS.mkdir(parents=True, exist_ok=True)
f = cfg["files"]

# Sub-directories follow Wildfire Commons naming convention
SURF = RAW / f"{TOWN}s-surface-fuels-and-surface-data" if TOWN == "prairie" else RAW / f"{TOWN}-surface-fuels-and-surface-data"
IGN_DIR  = RAW / f"{TOWN}-ignition"
WX_DIR   = RAW / f"{TOWN}-weather-data"
BLDG_DIR = RAW / f"{TOWN}-building-data"

# Sanity check
checks = {
    "SB40 fuel raster":  SURF     / f["sb40"],
    "Elevation raster":  SURF     / f["elevation"],
    "Depth (CBH)":       SURF     / f["depth"],
    "Moist 1hr":         SURF     / f["moist1"],
    "Rhof 1hr (CBD)":    SURF     / f["rhof1"],
    "SAV raster":        SURF     / f["sav"],
    "Ignition GeoJSON":  IGN_DIR  / f["ignition"],
    "Weather CSV":       WX_DIR   / f["weather"],
    "Buildings GeoJSON": BLDG_DIR / f["buildings"],
}

all_ok = True
print(f"Town: {cfg['display_name']}")
print(f"{'Resource':<25} {'Status'}")
print('─' * 50)
for name, path in checks.items():
    ok = path.exists()
    print(f"  {'✓' if ok else '✗'}  {name:<23} {path.name}")
    if not ok: all_ok = False

print()
print("✓ Cell 1 ready" if all_ok else "✗ Fix missing paths before continuing")

In [ ]:
# ============================================================
# CELL 2 — Raster metadata table
# ============================================================
tifs = {
    "SB40 (fuel model)": SURF / f["sb40"],
    "Elevation":          SURF / f["elevation"],
    "Depth (CBH)":        SURF / f["depth"],
    "Moist 1hr":          SURF / f["moist1"],
    "Moist 10hr":         SURF / f["moist10"],
    "Moist 100hr":        SURF / f["moist100"],
    "Rhof 1hr (CBD)":     SURF / f["rhof1"],
    "Rhof 10hr":          SURF / f["rhof10"],
    "Rhof 100hr":         SURF / f["rhof100"],
    "SAV":                SURF / f["sav"],
}

print(f"{'Layer':<22} {'CRS':<8} {'Shape':<14} {'Res (deg)':<12} {'Min':>8} {'Max':>8}")
print('─' * 82)
for name, path in tifs.items():
    with rasterio.open(path) as ds:
        d  = ds.read(1).astype(float)
        nd = ds.nodata
        if nd is not None: d = d[d != nd]
        d = d[d > -1e10]
        crs   = str(ds.crs).split(':')[-1]
        shape = f"{ds.height}\u00d7{ds.width}"
        res   = f"{ds.res[0]:.6f}"
        print(f"  {name:<20} {crs:<8} {shape:<14} {res:<12} "
              f"{d.min():>8.2f} {d.max():>8.2f}")

print("\n✓ Cell 2 ready — raster metadata inspected")

In [ ]:
# ============================================================
# CELL 3 — Weather, ignition point, buildings
# ============================================================
station_id = cfg["weather"]["station_id"]

# Weather
df_wx = pd.read_csv(WX_DIR / f["weather"], comment="#")
if station_id:
    df_wx = df_wx[df_wx.iloc[:, 0].astype(str).str.startswith(str(station_id), na=False)].copy()
df_wx.columns = ["Station_ID", "Date_Time", "temp_F", "RH", "ws_mph", "WD", "gust_mph"]
for col in ["temp_F", "RH", "ws_mph", "WD", "gust_mph"]:
    df_wx[col] = pd.to_numeric(df_wx[col], errors="coerce")
df_wx["Date_Time"] = pd.to_datetime(df_wx["Date_Time"]).dt.tz_localize(None)

interval_min = int(df_wx["Date_Time"].diff().dropna().dt.total_seconds().median() / 60)
print("── Weather ──────────────────────────────────────────")
print(f"  Station:  {df_wx['Station_ID'].iloc[0]}")
print(f"  Period:   {df_wx['Date_Time'].iloc[0]} → {df_wx['Date_Time'].iloc[-1]}")
print(f"  Rows:     {len(df_wx)}  (interval ≈ {interval_min} min)")
print(f"\n  {'Variable':<15} {'Min':>8} {'Max':>8} {'Mean':>8}")
print(f"  {'─'*42}")
for col, label in [("temp_F","Temp (°F)"), ("RH","RH (%)"),
                   ("ws_mph","Wind (mph)"), ("WD","Dir (°)"), ("gust_mph","Gust (mph)")]:
    print(f"  {label:<15} {df_wx[col].min():>8.1f} {df_wx[col].max():>8.1f} {df_wx[col].mean():>8.1f}")

# Ignition
with open(IGN_DIR / f["ignition"]) as fh:
    ign = json.load(fh)
feat = ign["features"][0]
lon, lat = feat["geometry"]["coordinates"]
start    = feat["properties"].get("start", "not specified")
print(f"\n── Ignition ─────────────────────────────────────────")
print(f"  Coordinates: {lat:.4f}°N, {abs(lon):.4f}°W")
print(f"  Start time:  {start}")

# Buildings
bldgs_raw = gpd.read_file(BLDG_DIR / f["buildings"])
print(f"\n── Buildings ────────────────────────────────────────")
print(f"  Features: {len(bldgs_raw):,}")
print(f"  CRS:      {bldgs_raw.crs}")
print(f"  Columns:  {list(bldgs_raw.columns)}")

print("\n✓ Cell 3 ready — weather, ignition, buildings explored")

In [ ]:
# ============================================================
# CELL 4 — 12-panel visualisation of all raw inputs
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(22, 16))
fig.suptitle(
    f"Raw Input Data Exploration — {cfg['display_name']}\n"
    "(rasters in native EPSG:4326 | slope/aspect derived for visualisation only)",
    fontsize=14, fontweight="bold"
)

def load_tif(path, nodata_fill=np.nan):
    with rasterio.open(path) as ds:
        d = ds.read(1).astype(float)
        if ds.nodata is not None:
            d[d == ds.nodata] = nodata_fill
        d[d < -1e10] = nodata_fill
        ext = [ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top]
    return d, ext

# 1. SB40
ax = axes[0, 0]
sb40, ext = load_tif(SURF / f["sb40"])
sb40[sb40 <= 0] = np.nan
sb40[sb40 == 32767] = np.nan
im = ax.imshow(sb40, extent=ext, origin="upper", cmap="tab20", interpolation="nearest")
plt.colorbar(im, ax=ax, label="S&B code", fraction=0.04)
ax.set_title("SB40 Fuel Model", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 2. Elevation
ax = axes[0, 1]
elev, ext_e = load_tif(SURF / f["elevation"])
im = ax.imshow(elev, extent=ext_e, origin="upper", cmap="terrain", interpolation="bilinear")
plt.colorbar(im, ax=ax, label="Elevation (m)", fraction=0.04)
ax.set_title("Elevation", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 3. Slope (derived)
ax = axes[0, 2]
with rasterio.open(SURF / f["elevation"]) as ds:
    res_deg = ds.res[0]
cell_m = res_deg * 111000
dz_dx = sobel(np.nan_to_num(elev), axis=1) / (8 * cell_m)
dz_dy = sobel(np.nan_to_num(elev), axis=0) / (8 * cell_m)
slope_viz = np.sqrt(dz_dx**2 + dz_dy**2) * 100
slope_viz[np.isnan(elev)] = np.nan
im = ax.imshow(slope_viz, extent=ext_e, origin="upper",
               cmap="YlOrRd", interpolation="nearest", vmin=0, vmax=60)
plt.colorbar(im, ax=ax, label="Slope (%)", fraction=0.04)
ax.set_title("Slope (derived)", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 4. Aspect (derived)
ax = axes[0, 3]
aspect_viz = (np.degrees(np.arctan2(-dz_dy, dz_dx)) + 360) % 360
aspect_viz[np.isnan(elev)] = np.nan
im = ax.imshow(aspect_viz, extent=ext_e, origin="upper",
               cmap="hsv", interpolation="nearest", vmin=0, vmax=360)
plt.colorbar(im, ax=ax, label="Aspect (°)", fraction=0.04)
ax.set_title("Aspect (derived)", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 5. Canopy Base Height
ax = axes[1, 0]
depth, ext_d = load_tif(SURF / f["depth"])
depth[depth == 0] = np.nan
im = ax.imshow(depth, extent=ext_d, origin="upper", cmap="Greens", interpolation="nearest")
plt.colorbar(im, ax=ax, label="CBH (m)", fraction=0.04)
ax.set_title("Canopy Base Height", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 6. Canopy Bulk Density
ax = axes[1, 1]
cbd, ext_c = load_tif(SURF / f["rhof1"])
cbd[cbd == 0] = np.nan
im = ax.imshow(cbd, extent=ext_c, origin="upper", cmap="YlOrBr", interpolation="nearest")
plt.colorbar(im, ax=ax, label="CBD kg/m³", fraction=0.04)
ax.set_title("Canopy Bulk Density", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 7. 1hr Fuel Moisture
ax = axes[1, 2]
moist1, ext_m = load_tif(SURF / f["moist1"])
im = ax.imshow(moist1 * 100, extent=ext_m, origin="upper",
               cmap="RdYlGn", interpolation="nearest")
plt.colorbar(im, ax=ax, label="FMC 1hr (%)", fraction=0.04)
ax.set_title("1hr Fuel Moisture", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 8. SAV
ax = axes[1, 3]
sav, ext_s = load_tif(SURF / f["sav"])
sav[sav == 0] = np.nan
im = ax.imshow(sav, extent=ext_s, origin="upper", cmap="plasma", interpolation="nearest")
plt.colorbar(im, ax=ax, label="SAV (1/m)", fraction=0.04)
ax.set_title("Surface Area-to-Volume Ratio", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# 9. Wind speed + RH
ax = axes[2, 0]
ax2 = ax.twinx()
ax.plot(df_wx["Date_Time"], df_wx["ws_mph"], color="#d73027", linewidth=2, label="Wind (mph)")
ax.plot(df_wx["Date_Time"], df_wx["gust_mph"], color="#fc8d59", linewidth=1.5,
        linestyle="--", label="Gust (mph)")
ax2.plot(df_wx["Date_Time"], df_wx["RH"], color="#4575b4", linewidth=2, label="RH (%)")
ax.set_ylabel("Wind (mph)", color="#d73027", fontsize=8)
ax2.set_ylabel("RH (%)", color="#4575b4", fontsize=8)
ax.set_title("Wind Speed & Humidity", fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=7)
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=7, loc="upper left")
ax.grid(True, alpha=0.3)

# 10. Wind direction
ax = axes[2, 1]
ax.plot(df_wx["Date_Time"], df_wx["WD"], color="#542788", linewidth=2)
ax.set_ylabel("Wind Direction (°)")
ax.set_title("Wind Direction", fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=7)
ax.set_ylim(0, 360)
ax.axhline(225, color="grey", linestyle=":", alpha=0.5, label="SW (225°)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 11. Temperature
ax = axes[2, 2]
ax.plot(df_wx["Date_Time"], df_wx["temp_F"], color="#e08214", linewidth=2)
ax.set_ylabel("Temperature (°F)")
ax.set_title("Temperature", fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=7)
ax.grid(True, alpha=0.3)

# 12. Buildings + ignition
ax = axes[2, 3]
ax.imshow(sb40, extent=ext, origin="upper", cmap="YlGn", alpha=0.6, interpolation="nearest")
bldgs_raw.plot(ax=ax, color="steelblue", markersize=1.5, alpha=0.6, zorder=3)
ax.plot(lon, lat, "*", color="red", markersize=16,
        markeredgecolor="black", markeredgewidth=1, zorder=5, label="Ignition")
ax.set_xlim(ext[0], ext[1])
ax.set_ylim(ext[2], ext[3])
ax.legend(fontsize=8)
ax.set_title("Buildings & Ignition Point", fontweight="bold")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

plt.tight_layout()
out = PLOTS / "01_raw_data_exploration.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Cell 4 ready — saved to {out}")